# Data Quality Audit of Raw Data from the Investing.com Data Platform

The main objective of the data quality audit is to assess the quality, consistency, and general characteristics of the raw data obtained from the Investing.com.

The analysis focuses on the following aspects:
* Missing data and missing observations
* Index integrity and consistency
* Duplicated indices
* Timestamp and time zone consistency
* Overall data characteristics

**All data and variables fetched from Investing.com are explained in the [Investing.com data overview](./01_Investing_com_data_overview.ipynb).**

In [1]:
import sys
from pathlib import Path

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sys.path.append(str(Path.cwd().parent.parent))
from src.config import (
    DATA_DIR,
)
from src.eda.validation import (
    print_index_summary,
    print_missing_values,
    find_missing_dates,
    print_data_overview,
)

In [2]:
data_dir = DATA_DIR / "raw" / "investing_com"
data_inv = {}
for file in data_dir.iterdir():
    data = pd.read_csv(file)
    data.index = pd.to_datetime(data["Date"], utc=True)
    key = file.stem
    data_inv[key] = data

## 1. Missing Data

In [3]:
for name, data in data_inv.items():
    print_missing_values(data=data, name=name)


--- [eur_pln] Missing values ---
Vol.    990

--- [eu_carbon_emissions_futures] Missing values ---
Vol.    3

--- [german_power_baseload_futures] Missing values ---
Vol.    941

--- [rotterdam_coal_futures] Missing values ---
Vol.    210

--- [dutch_ttf_natural_gas_futures] Missing values ---
Vol.    1


## Missing Data – Observations

The amount of missing data varies substantially across the datasets. The **EUR/PLN** dataset contains the highest number of missing observations (**990**), followed by **German power baseload futures** (**941**) and **Rotterdam coal futures** (**210**).

In contrast, the **EU carbon emissions futures** and **Dutch TTF natural gas futures** datasets contain very few missing observations, with **3** and **1** missing values, respectively.

Overall, missing observations are concentrated primarily in the **EUR/PLN** and **German power baseload futures** datasets. However, the presence of missing values needs to be considered together with the structure of the time index, since some apparent gaps may correspond to non-trading days rather than 

## 2. Index Integrity

In [4]:
for name, data in data_inv.items():
    print_index_summary(data=data, name=name)


--- [eur_pln] Index summary ---
Duplicates: 0
Timezone:  UTC

Time steps:
  -1 days +00:00:00: 791
  -3 days +00:00:00: 198

--- [eu_carbon_emissions_futures] Index summary ---
Duplicates: 0
Timezone:  UTC

Time steps:
  -1 days +00:00:00: 770
  -3 days +00:00:00: 191
  -5 days +00:00:00: 6
  -2 days +00:00:00: 2
  -4 days +00:00:00: 2

--- [german_power_baseload_futures] Index summary ---
Duplicates: 0
Timezone:  UTC

Time steps:
  -1 days +00:00:00: 730
  -3 days +00:00:00: 166
  -4 days +00:00:00: 30
  -2 days +00:00:00: 12
  -7 days +00:00:00: 1
  -6 days +00:00:00: 1

--- [rotterdam_coal_futures] Index summary ---
Duplicates: 0
Timezone:  UTC

Time steps:
  -1 days +00:00:00: 775
  -3 days +00:00:00: 190
  -4 days +00:00:00: 8
  -2 days +00:00:00: 4

--- [dutch_ttf_natural_gas_futures] Index summary ---
Duplicates: 0
Timezone:  UTC

Time steps:
  -1 days +00:00:00: 793
  -3 days +00:00:00: 169
  -2 days +00:00:00: 24
  -5 days +00:00:00: 5
  -4 days +00:00:00: 3


In [5]:
for name, data in data_inv.items():
    missing_index = find_missing_dates(data=data, name=name)
    print("missing indicies for Monday-Friday")
    print([i for i in missing_index if i.day_of_week < 5])


--- [eur_pln] Missing dates ---
Count: 396

Missing ranges:
  2022-12-03 00:00:00+00:00 - 2022-12-04 00:00:00+00:00
  2022-12-10 00:00:00+00:00 - 2022-12-11 00:00:00+00:00
  2022-12-17 00:00:00+00:00 - 2022-12-18 00:00:00+00:00
  2022-12-24 00:00:00+00:00 - 2022-12-25 00:00:00+00:00
  2022-12-31 00:00:00+00:00 - 2023-01-01 00:00:00+00:00
  2023-01-07 00:00:00+00:00 - 2023-01-08 00:00:00+00:00
  2023-01-14 00:00:00+00:00 - 2023-01-15 00:00:00+00:00
  2023-01-21 00:00:00+00:00 - 2023-01-22 00:00:00+00:00
  2023-01-28 00:00:00+00:00 - 2023-01-29 00:00:00+00:00
  2023-02-04 00:00:00+00:00 - 2023-02-05 00:00:00+00:00
  2023-02-11 00:00:00+00:00 - 2023-02-12 00:00:00+00:00
  2023-02-18 00:00:00+00:00 - 2023-02-19 00:00:00+00:00
  2023-02-25 00:00:00+00:00 - 2023-02-26 00:00:00+00:00
  2023-03-04 00:00:00+00:00 - 2023-03-05 00:00:00+00:00
  2023-03-11 00:00:00+00:00 - 2023-03-12 00:00:00+00:00
  2023-03-18 00:00:00+00:00 - 2023-03-19 00:00:00+00:00
  2023-03-25 00:00:00+00:00 - 2023-03-26 00

### Index Integrity - observations

All datasets have **no duplicated indices**, and all indices are expressed in **UTC**, indicating that there are no issues with duplicated timestamps or timezone inconsistencies.

The **EUR/PLN** and **EU carbon emissions futures** datasets exhibit a relatively regular index structure, with **1-day** time steps dominating and only a small number of **3-day** gaps.

The **German power baseload futures** dataset has the most irregular index structure, with time steps ranging from **1 to 7 days** and several **2-, 3-, and 4-day** gaps.

The **Rotterdam coal futures** and **Dutch TTF natural gas futures** datasets are primarily characterized by **1-day and 3-day** time steps, with only a small number of longer gaps.

Overall, the indices are free of duplicates and timezone inconsistencies. The longer time steps are concentrated mainly in the **German power baseload futures** dataset and may reflect non-trading days


## 3. Data statistics and properties

In [6]:
for name, data in data_inv.items():
    print_data_overview(data=data, name=name)


--- [eur_pln] Overview ---
Shape: 990 x 7

Info:
<class 'pandas.DataFrame'>
DatetimeIndex: 990 entries, 2026-09-16 00:00:00+00:00 to 2022-12-01 00:00:00+00:00
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Date      990 non-null    str    
 1   Price     990 non-null    float64
 2   Open      990 non-null    float64
 3   High      990 non-null    float64
 4   Low       990 non-null    float64
 5   Vol.      0 non-null      float64
 6   Change %  990 non-null    str    
dtypes: float64(5), str(2)
memory usage: 76.9 KB

Description:
            Price        Open        High         Low  Vol.
count  990.000000  990.000000  990.000000  990.000000   0.0
mean     4.347952    4.350618    4.365143    4.338543   NaN
std      0.148615    0.150000    0.151906    0.147946   NaN
min      4.136300    4.138800    4.150000    4.126300   NaN
25%      4.252125    4.252825    4.265300    4.244325   NaN
50%      4.289400    4.292100    4

In [7]:
print("Coefficient of Variation (CV)")
for name, data in data_inv.items():
    cv = data["Price"].std() / data["Price"].mean()
    print(f"dataset: {name}, price CV = {cv.round(2)}")

Coefficient of Variation (CV)
dataset: eur_pln, price CV = 0.03
dataset: eu_carbon_emissions_futures, price CV = 0.12
dataset: german_power_baseload_futures, price CV = 0.36
dataset: rotterdam_coal_futures, price CV = 0.21
dataset: dutch_ttf_natural_gas_futures, price CV = 0.38


## 3. Conclusions and data preprocessing decisions

Based on the missing-data and index-integrity analysis, the following preprocessing decisions will be applied:

**3.1** No duplicated indices or timezone inconsistencies were identified. Therefore, **no deduplication or timezone conversion is required**.

**3.2** Gaps corresponding to **non-trading days** will not be treated as missing observations. Where a continuous index is required for alignment across datasets, these gaps will be handled using **forward fill (FFILL)** rather than linear interpolation. This approach preserves the last available market value without introducing artificial price movements.

**3.3** Missing observations that occur on **expected trading days** will be treated separately from non-trading-day gaps. They will **not be automatically forward-filled**, as doing so could incorrectly imply that the market price remained unchanged when no observation was actually recorded.

This distinction between **structural gaps caused by non-trading days** and **genuine missing observations on trading days** will therefore guide the subsequent data preprocessing.